# Lab 3: Image generation

## Overview

The endpoint also generates images. It runs the FLUX.2 klein 4B model, a distilled model that produces an image in four steps: about one second at 512 pixels and about four seconds at 1,024 pixels on the event hardware.

The API follows the OpenAI images API, with a deliberately small surface: one square image per request, returned as base64 PNG data. This lab covers a first image, sizes, reproducible results with seeds, and the per-team quota.

## Learning objectives

By the end of this lab you will be able to:

- Generate an image and display or save it
- Choose a size and know the time cost of each
- Reproduce or vary an image with a seed
- Handle the image quota and a busy queue without failing your app

## Prerequisites

- Lab 1 completed, including the `.env` file
- About 15 minutes

## Step 1: Set up your environment

In [ ]:
# Installs the packages this lab needs. Safe to rerun.
%pip install -q "openai>=3,<4" python-dotenv pillow

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Reads OPENAI_BASE_URL and OPENAI_API_KEY from the .env file in the repository root.
# override=True makes the file win over any OPENAI_API_KEY already set in your shell.
load_dotenv(Path("..") / ".env", override=True)
load_dotenv(override=True)

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
print("Endpoint:", base_url)
print("Key:     ", api_key[:6] + "..." + api_key[-4:])

client = OpenAI()  # The SDK reads the same two variables, so no arguments are needed.

## The call in one look

`POST {OPENAI_BASE_URL}/images/generations` with header `Authorization: Bearer <team key>` and a JSON body.

| Field | Required | Values and defaults |
|---|---|---|
| `model` | yes | `"hackathon-image"` |
| `prompt` | yes | text, up to 2,000 characters |
| `size` | no | `"512x512"`, `"768x768"`, or `"1024x1024"`; default 1024 |
| `n` | no | always 1 |
| `response_format` | no | always `"b64_json"` |
| `extra_body` | no | `{"seed": 7}` for a different or repeatable result; default seed 0 |

The response has `data[0].b64_json`, the PNG as a base64 string. The minimal call:

In [ ]:
import base64

response = client.images.generate(
    model="hackathon-image",
    prompt="A yellow bicycle, simple icon",
    size="512x512",
    response_format="b64_json",
)
png_bytes = base64.b64decode(response.data[0].b64_json)
open("minimal.png", "wb").write(png_bytes)
print("Saved minimal.png,", len(png_bytes), "bytes")

## Step 2: Generate your first image

The model alias is `hackathon-image`. The response carries the PNG as a base64 string in `b64_json`. Decode it, open it with Pillow, and it is an ordinary image you can display, save, or send to a browser.

In [ ]:
import base64
import io
import time
from IPython.display import display
from PIL import Image

def to_image(response):
    """Decode the first image in an images.generate response."""
    return Image.open(io.BytesIO(base64.b64decode(response.data[0].b64_json)))

start = time.time()
response = client.images.generate(
    model="hackathon-image",
    prompt="A friendly robot holding a coffee cup, flat vector illustration, warm colors",
    size="512x512",
    n=1,
    response_format="b64_json",
)
print(f"Generated in {time.time() - start:.1f}s | fields: {list(response.model_dump().keys())}")

image = to_image(response)
print("Size:", image.size, "| mode:", image.mode)
image.save("lab3-first.png")
display(image)

## Step 3: Pick a size

Three sizes are supported, all square: `512x512`, `768x768`, and `1024x1024`. Larger takes longer and the queue is shared by every team, so use 512 while you iterate on a prompt and 1,024 for the final result.

In [ ]:
prompt = "A lighthouse on a rocky coast at dusk, oil painting"

for size in ("768x768", "1024x1024"):
    start = time.time()
    r = client.images.generate(model="hackathon-image", prompt=prompt, size=size, response_format="b64_json")
    img = to_image(r)
    print(f"{size}: {time.time() - start:.1f}s, {img.size}")
    img.save(f"lab3-{size}.png")

display(img)

## Step 4: Know what the endpoint rejects

Anything outside the supported surface is rejected with `422` before it reaches a GPU: more than one image, a non-square or unlisted size, `response_format="url"`, or an unknown field. The request still counts against your quota, so check parameters in your code rather than relying on the error.

In [ ]:
import openai

try:
    client.images.generate(model="hackathon-image", prompt="a red cube", size="512x512", response_format="url")
except openai.APIStatusError as e:
    print("Status:", e.status_code)
    print("Body:  ", e.response.text[:160])

## Step 5: Reproduce or vary a result with a seed

Generation is deterministic for a given prompt, size, and seed. The default seed is 0, which means the same prompt returns the same picture every time. That is useful while you refine wording, and surprising when you want variety.

`seed` is not part of the OpenAI API, so pass it through `extra_body`. Any integer from 0 to 2,147,483,647 works.

In [ ]:
prompt = "A small wooden cabin in a snowy forest, morning light, watercolor"
images = []
for seed in (7, 42):
    r = client.images.generate(model="hackathon-image", prompt=prompt, size="512x512",
                               response_format="b64_json", extra_body={"seed": seed})
    images.append(to_image(r))

side_by_side = Image.new("RGB", (1024 + 16, 512), "white")
side_by_side.paste(images[0], (0, 0))
side_by_side.paste(images[1], (512 + 16, 0))
display(side_by_side)

## Step 6: Handle the quota and the queue

Two protections sit in front of the image model:

| Protection | Response |
|---|---|
| More than 6 image requests per team per minute | `429` with `x-ratelimit-reset` seconds |
| More than 4 requests waiting for the single GPU | `429` with `Retry-After: 10` |

Both are the same status code, so one handler covers both: read whichever header is present, wait that long plus a little randomness, and try again. You have already made six requests in this lab, so the next one may be rejected. The cell shows the rejection and the recovery.

In [ ]:
import random

quota_client = OpenAI(max_retries=0)  # Retries are handled here so you can see them.

def generate_with_retry(prompt, size="512x512", seed=0, attempts=4):
    for attempt in range(attempts):
        try:
            return quota_client.images.generate(
                model="hackathon-image", prompt=prompt, size=size,
                response_format="b64_json", extra_body={"seed": seed},
            )
        except openai.RateLimitError as e:
            headers = e.response.headers
            wait = int(headers.get("retry-after") or headers.get("x-ratelimit-reset") or 10)
            wait += random.uniform(0, 2)
            print(f"429 received (limit {headers.get('x-ratelimit-limit', '?')}/min); waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError("Gave up after repeated 429 responses")

r = generate_with_retry("A paper airplane over a city skyline, minimal poster")
print("Succeeded.")
display(to_image(r))

If no `429` appeared, the minute window rolled over between steps. The handler is still what you want in any app that generates images in a loop or for several users.

## Things to know

- **Prompts.** Name the subject, the style, and the lighting in one or two sentences. This model does not use negative prompts; say what you want, not what you do not want.
- **One image per request, PNG only.** Compose grids or variations on your side, as Step 5 did.
- **Cost of a request.** About 1 second at 512, 2 seconds at 768, 4 seconds at 1,024, on a GPU shared by every team.
- **No image input.** The chat model does not accept images on this deployment, and the image endpoint has no edit or variation operation.

## Validation

In [ ]:
raw = base64.b64decode(response.data[0].b64_json)
assert raw[:8] == b"\x89PNG\r\n\x1a\n", "not a PNG"
assert image.size == (512, 512)
print("Lab 3 complete.")

## Clean up

The images were saved next to this notebook. Remove them if you do not want them in your repository.

In [ ]:
from pathlib import Path

for path in list(Path(".").glob("lab3-*.png")) + list(Path(".").glob("minimal.png")):
    path.unlink()
    print("Removed", path)

## Next steps

You now have all three endpoints working from Python: chat for reasoning and writing, embeddings for search over your own data, and images for visuals. Lab 4 adds the two features that turn a chat model into the core of an application: tool calling and structured output.